Exploratory notebook: final presentation figures are generated by `python3 run.py` and recorded in `analsis/current/results.json`. Some narrative counts below predate the final date-filter, pricing and cost changes. Use the current report for claims; run the raw workflow and download with `--all` before using reviews/calendars.

# Where and what to buy

**Brief: Where (neighbourhoods) + What (property types and mix).**

- Revenue, nightly price and occupancy by neighbourhood and type
- Which segments have the best yield once purchase price is included
- Watch out: segments with few listings (`n`) have unreliable medians

In [1]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `seg`: neighbourhood × type × bedrooms (active entire homes)
- `uw`: segments + purchase price, costs, NOI, net yield

### From 01 (keep in mind)
- **Occupancy & revenue** = Inside Airbnb's estimates (`estimated_occupancy_l365d`, `estimated_revenue_l365d`), 12 months to 15 June 2026. Estimates, not real bookings.
- **Price** = one night's quote around June–July 2026 (World Cup period), so revenue may be a bit high.
- **`is_active`** = reviewed in the last 12 months. Only active listings are used for segments.
- Cleaned data: 6,509 → 4,447 listings (no price, price outside $30–$2,000, 90+ night minimum removed: BC/Vancouver short-term = under 90 nights).

In [2]:
seg = pd.read_csv(T / "segments.csv", dtype={"bedrooms_grp": str})
uw = pd.read_csv(T / "segments_underwritten.csv", dtype={"bedrooms_grp": str})

## Quick look

In [3]:
uw[uw["n"] >= 5].sort_values("net_yield", ascending=False).head(20)

,neighbourhood_cleansed,bucket,bedrooms_grp,n,adr_median,occ_median,rev_median,rev_p25,rev_p75,rating,share_commercial,share_licensed_est,prop_type,median_value,assessment_properties,price_cad,gross,fixed,cash_in,noi,net_yield
14,Downtown Eastside,condo,3+,12.00,509.50,0.31,"73,950.00","20,284.50","115,263.00",4.67,0.50,0.83,condo,"617,500.00","4,248.00","926,250.00","73,950.00","12,278.75","988,300.00","29,503.00",0.03
12,Downtown Eastside,condo,1,146.00,368.16,0.35,"49,140.00","16,682.25","75,913.50",4.89,0.14,0.95,condo,"617,500.00","4,248.00","524,875.00","49,140.00","11,074.62","570,870.00","16,689.48",0.03
7,Downtown,condo,2,288.00,541.38,0.33,"63,091.50","24,832.00","101,596.50",4.88,0.18,0.98,condo,"758,200.00","37,457.00","871,930.00","63,091.50","12,115.79","931,807.20","23,530.91",0.03
8,Downtown,condo,3+,116.00,575.08,0.33,"75,922.50","28,647.00","124,740.75",4.82,0.38,0.93,condo,"758,200.00","37,457.00","1,137,300.00","75,922.50","12,911.90","1,207,792.00","29,984.31",0.02
6,Downtown,condo,1,365.00,410.00,0.35,"50,760.00","20,556.00","82,176.00",4.88,0.15,0.96,condo,"758,200.00","37,457.00","644,470.00","50,760.00","11,433.41","695,248.80","17,245.99",0.02
109,West End,condo,2,23.00,513.53,0.28,"53,466.00","24,519.00","73,128.00",4.91,0.17,0.96,condo,"779,636.00","9,190.00","896,581.40","53,466.00","12,189.74","957,444.66","18,018.55",0.02
17,Downtown Eastside,house,3+,8.00,734.75,0.22,"65,160.00","40,914.75","103,031.25",4.84,0.12,1.00,house,"1,566,000.00",536.00,"1,566,000.00","65,160.00","8,198.00","1,653,640.00","28,617.40",0.02
13,Downtown Eastside,condo,2,79.00,542.06,0.25,"43,456.00","15,560.00","82,260.00",4.90,0.27,0.92,condo,"617,500.00","4,248.00","710,125.00","43,456.00","11,630.38","763,530.00","12,922.27",0.02
54,Kitsilano,condo,1,31.00,380.00,0.31,"47,511.00","14,269.00","68,586.00",4.88,0.13,1.00,condo,"988,900.00","7,680.00","840,565.00","47,511.00","12,021.69","899,187.60","14,822.02",0.02
40,Kensington-Cedar Cottage,condo,1,5.00,310.40,0.33,"46,050.00","44,095.00","52,500.00",4.76,0.20,1.00,condo,"1,123,000.00","2,949.00","954,550.00","46,050.00","12,363.65","1,017,732.00","13,654.60",0.01


In [4]:
uw.groupby("bucket")[["rev_median", "price_cad", "net_yield"]].median()

,rev_median,price_cad,net_yield
bucket,,,
condo,"32,483.00","1,079,850.00",0.01
house,"36,328.50","2,601,075.00",0.00


## Analysis
_Start here._

### Step 1: Demand by neighbourhood
Active entire-home listings only (what DDD would run). Typical (median) value per neighbourhood. Hover a bar to see all the numbers for that neighbourhood.

In [5]:
import altair as alt

df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)
d = df[df["is_active"] & (df["room_type"] == "Entire home/apt")]

hood = (d.groupby("neighbourhood_cleansed")
          .agg(listings=("id", "size"), revenue=("revenue", "median"), total_revenue=("revenue", "sum"),
               nights_booked=("estimated_occupancy_l365d", "median"), price=("price_num", "median"))
          .reset_index())
hood

,neighbourhood_cleansed,listings,revenue,total_revenue,nights_booked,price
0,Arbutus Ridge,64,"38,682.00","3,579,934.00",108.00,400.00
1,Downtown,798,"58,486.00","51,770,098.00",126.00,460.97
2,Downtown Eastside,290,"44,043.00","15,640,536.00",102.00,401.18
3,Dunbar Southlands,125,"32,886.00","5,417,985.00",66.00,385.72
4,Fairview,18,"45,239.00","1,114,288.00",96.00,435.75
5,Grandview-Woodland,128,"36,199.50","5,402,280.00",93.00,350.88
6,Hastings-Sunrise,176,"42,266.50","8,842,193.00",123.00,312.88
7,Kensington-Cedar Cottage,217,"40,741.00","10,291,055.00",120.00,350.00
8,Kerrisdale,46,"32,396.50","2,093,233.00",72.00,399.98
9,Killarney,73,"24,960.00","2,864,865.00",78.00,319.53


In [6]:
def bar(col, title, fmt, note=""):
    tip = [alt.Tooltip("neighbourhood_cleansed:N", title="Neighbourhood"),
           alt.Tooltip("revenue:Q", title="Typical yearly revenue", format="$,.0f"),
           alt.Tooltip("nights_booked:Q", title="Typical nights booked / year", format=",.0f"),
           alt.Tooltip("price:Q", title="Typical nightly price", format="$,.0f"),
           alt.Tooltip("listings:Q", title="Listings")]
    chart = (alt.Chart(hood)
             .mark_bar(color="#2a78d6", cornerRadiusEnd=4, height=16)
             .encode(x=alt.X(f"{col}:Q", title=None, axis=alt.Axis(format=fmt, grid=True, gridColor="#eee")),
                     y=alt.Y("neighbourhood_cleansed:N", sort="-x", title=None),
                     tooltip=tip)
             .properties(width=550, height=alt.Step(20),
                         title=alt.Title(title, subtitle=note, anchor="start", subtitleColor="#52514e"))
             .configure_view(stroke=None))
    display(chart)

bar("revenue", "Typical yearly revenue per listing (CAD)", "$,.0f",
    note="Yearly revenue = nights booked × nightly price · 12 months to June 2026")
bar("nights_booked", "Typical occupancy per year", ",.0f",
    note="Occupancy = nights booked out of 365 · 12 months to June 2026")
bar("listings", "Number of active whole-home listings", ",d",
    note="Supply: more listings = proven demand, but also more competition")
bar("total_revenue", "Total yearly revenue, all active whole-home listings (CAD)", "$,.0f",
    note="Size of the market = sum of every listing's revenue · 12 months to June 2026")
bar("price", "Typical nightly price (CAD)", "$,.0f",
    note="Quote for one night around June–July 2026 (World Cup period)")

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### Step 2: Return per neighbourhood → top 5
**Gross yield = estimated revenue ÷ purchase price**
- Revenue = nights booked × nightly price (Inside Airbnb estimate, 12 months to June 2026)
- Purchase price = median 2026 assessed value for that neighbourhood × type (City of Vancouver property tax report), condos scaled by bedrooms

Operating costs are left out here: they shrink every neighbourhood about equally, so the ranking doesn't change (tested: same top 5, rank agreement 0.99). They come back only for the final 2027 profit number.

Filter 1: at least 50 listings (reliable numbers). Filter 2: rank by typical gross yield. Suites excluded (can't buy without the house). Strathcona has no homes in the tax data, so it drops out.

In [7]:
from src import portfolio

d = d.copy()
d["bedrooms_grp"] = d["bedrooms"].fillna(1).clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
d_ret = portfolio.add_purchase_prices(d)                      # one row per listing, adds price_cad
d_ret["gross_yield"] = d_ret["revenue"] / d_ret["price_cad"]

ret = (d_ret.groupby("neighbourhood_cleansed")
            .agg(listings=("id", "size"), revenue=("revenue", "median"),
                 purchase_price=("price_cad", "median"), gross_yield=("gross_yield", "median"))
            .reset_index())
shortlist = ret[ret["listings"] >= 50].sort_values("gross_yield", ascending=False)
top5 = shortlist.head(5)
shortlist

,neighbourhood_cleansed,listings,revenue,purchase_price,gross_yield
1,Downtown,796,"58,486.00","871,930.00",0.07
2,Downtown Eastside,279,"44,472.00","524,875.00",0.07
20,West End,78,"41,136.00","662,690.60",0.05
12,Mount Pleasant,109,"33,886.00","859,165.00",0.04
7,Kensington-Cedar Cottage,137,"41,904.00","1,824,000.00",0.02
6,Hastings-Sunrise,112,"35,819.50","1,739,050.00",0.02
14,Renfrew-Collingwood,102,"33,180.00","1,786,000.00",0.02
10,Kitsilano,167,"43,368.00","2,761,500.00",0.02
5,Grandview-Woodland,87,"25,884.00","1,843,000.00",0.02
19,Victoria-Fraserview,93,"33,480.00","2,025,000.00",0.02


### Step 2b: Fair comparison — condos vs condos, houses vs houses
Step 2 mixes types: Downtown is mostly condos (cheap to buy), other areas mostly houses (expensive), so condos win by default. 
Here each type is ranked on its own. Same filter: at least 50 listings of that type.

In [8]:
by_type = (d_ret.groupby(["bucket", "neighbourhood_cleansed"])
                .agg(listings=("id", "size"), revenue=("revenue", "median"),
                     purchase_price=("price_cad", "median"), gross_yield=("gross_yield", "median"))
                .reset_index())
by_type = by_type[by_type["listings"] >= 50].sort_values(["bucket", "gross_yield"], ascending=[True, False])

for bucket, t in by_type.groupby("bucket"):
    print(f"\n=== {bucket.upper()} ===")
    print(t.drop(columns="bucket").to_string(index=False, formatters={
        "revenue": "${:,.0f}".format, "purchase_price": "${:,.0f}".format,
        "gross_yield": "{:.1%}".format}))


=== CONDO ===
neighbourhood_cleansed  listings revenue purchase_price gross_yield
     Downtown Eastside       249 $45,390       $524,875        7.6%
              Downtown       787 $58,644       $871,930        7.4%
              West End        74 $41,136       $662,691        5.8%
        Mount Pleasant        74 $29,886       $635,035        4.6%
             Kitsilano        51 $41,247       $840,565        3.6%

=== HOUSE ===
  neighbourhood_cleansed  listings revenue purchase_price gross_yield
Kensington-Cedar Cottage       127 $41,154     $1,824,000        2.3%
        Hastings-Sunrise       103 $35,999     $1,739,050        2.1%
     Renfrew-Collingwood        95 $36,378     $1,786,000        2.0%
      Grandview-Woodland        63 $33,696     $1,843,000        1.8%
               Kitsilano       116 $47,352     $2,761,500        1.7%
     Victoria-Fraserview        91 $33,480     $2,025,000        1.7%
                  Sunset        79 $27,936     $1,880,000        1.5%
  